# 📊 Data Analysis — 10-Week Course
## Week 6: Statistical Analysis & Hypothesis Testing

---
### Learning Objectives
By the end of this week, you will be able to:
- Understand probability distributions and their properties
- Formulate and test statistical hypotheses (t-test, chi-square, ANOVA)
- Interpret p-values and confidence intervals correctly
- Quantify effect size (Cohen's d) to complement significance testing

### Outline
1. Probability Distributions
2. The Hypothesis Testing Framework
3. t-Tests
4. Chi-Square Tests
5. One-Way ANOVA
6. Confidence Intervals
7. Effect Size
8. Lab Exercises
9. Assignment

---
## 1. Probability Distributions

### Key Distributions in Data Analysis
| Distribution | Shape | Common Use |
|---|---|---|
| **Normal** | Bell curve, symmetric | Test scores, measurement errors, heights |
| **Log-normal** | Right-skewed | Income, GDP, prices |
| **Exponential** | Right-skewed, memoryless | Waiting times, survival times |
| **Poisson** | Discrete, right-skewed | Count data (events per time period) |
| **Binomial** | Discrete | Yes/No outcomes (success/failure) |
| **Uniform** | Flat | Random number generation |

### Why Distribution Matters
- Determines which statistical test to use
- Affects interpretation of mean vs median
- Guides choice of visualisation
- Log-skewed data → apply log transform before parametric tests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (norm, lognorm, expon, poisson, binom, chi2,
                          ttest_1samp, ttest_ind, ttest_rel,
                          chi2_contingency, f_oneway, shapiro)
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

import os
if os.path.exists("africa_health_cleaned.csv"):
    df = pd.read_csv("africa_health_cleaned.csv")
else:
    np.random.seed(42)
    N = 54
    countries = [
        "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
        "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
        "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
        "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
        "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
        "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
        "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
        "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
    ]
    regions = (["West Africa"]*16+["East Africa"]*14+
               ["North Africa"]*6+["Central Africa"]*8+["Southern Africa"]*10)
    df = pd.DataFrame({
        "country"           : countries,
        "region"            : regions,
        "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
        "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
        "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
        "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
        "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
        "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
        "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
        "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    })
print("Dataset ready:", df.shape)

In [ ]:
# Common distributions illustrated
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
np.random.seed(0)

dist_data = [
    ("Normal μ=0, σ=1",    np.random.normal(0, 1, 2000),     "#3498DB"),
    ("Log-normal μ=0, σ=1", np.random.lognormal(0, 1, 2000), "#E74C3C"),
    ("Exponential λ=1",     np.random.exponential(1, 2000),   "#2ECC71"),
    ("Poisson λ=4",         np.random.poisson(4, 2000),        "#9B59B6"),
    ("Binomial n=20, p=0.3",np.random.binomial(20, 0.3, 2000),"#F39C12"),
    ("Uniform [0, 1]",      np.random.uniform(0, 1, 2000),    "#1ABC9C"),
]

for ax, (label, data, color) in zip(axes, dist_data):
    ax.hist(data, bins=30, color=color, edgecolor="white", alpha=0.8, density=True)
    ax.set_title(label, fontweight="bold", fontsize=9)
    ax.set_ylabel("Density")

fig.suptitle("Common Probability Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("week6_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Normality testing with Shapiro-Wilk
print("=== Shapiro-Wilk Normality Tests ===")
print(f"{'Variable':<25} {'W statistic':>12} {'p-value':>12} {'Normal?':>10}")
print("-" * 65)

num_cols = df.select_dtypes("number").columns.tolist()
for col in num_cols:
    data = df[col].dropna()
    # Shapiro-Wilk requires n ≤ 5000
    W, p = shapiro(data)
    is_normal = "Yes" if p > 0.05 else "No"
    print(f"{col:<25} {W:>12.4f} {p:>12.4f} {is_normal:>10}")

print("\nInterpretation: p > 0.05 → fail to reject normality (data may be normal)")

---
## 2. The Hypothesis Testing Framework

```
Step 1: State hypotheses
         H₀ (null)       — No effect / no difference
         H₁ (alternative) — Effect exists / difference exists

Step 2: Choose significance level α  (usually 0.05)

Step 3: Select and run the appropriate test

Step 4: Compute test statistic and p-value

Step 5: Decision rule
         p ≤ α → Reject H₀  (result is statistically significant)
         p > α → Fail to reject H₀

Step 6: Interpret in context + report effect size
```

### Common Errors
| Error | Meaning | Consequence |
|---|---|---|
| **Type I (α)** | Reject H₀ when it is true | False positive |
| **Type II (β)** | Fail to reject H₀ when it is false | False negative |

### p-Value Misconceptions
- ❌ p-value is NOT the probability that H₀ is true
- ❌ p-value is NOT the probability of the result being due to chance
- ✓ p-value = P(observing data this extreme | H₀ is true)

---
## 3. t-Tests

| Test | Use Case | H₀ |
|---|---|---|
| **One-sample t** | Compare sample mean to known value | μ = μ₀ |
| **Two-sample t (Welch's)** | Compare means of two independent groups | μ₁ = μ₂ |
| **Paired t** | Compare before/after measurements | μ_diff = 0 |

In [ ]:
def ttest_report(name, t_stat, p_val, n, alpha=0.05, ci=None, d=None):
    """Pretty-print a t-test result."""
    sig = "✓ Significant" if p_val <= alpha else "✗ Not significant"
    print(f"\n{'='*55}")
    print(f"TEST: {name}")
    print(f"  t = {t_stat:.4f},  p = {p_val:.4f},  n = {n}")
    if ci:
        print(f"  95% CI: [{ci[0]:.3f}, {ci[1]:.3f}]")
    if d is not None:
        strength = "small" if abs(d)<0.5 else "medium" if abs(d)<0.8 else "large"
        print(f"  Cohen's d = {d:.3f} ({strength} effect)")
    print(f"  Decision (α={alpha}): {sig}")

# ── One-sample t-test ──────────────────────────────────────────────────
# H₀: Mean life expectancy = 65 years
le = df["life_expectancy"]
mu_0 = 65.0
t, p = ttest_1samp(le, popmean=mu_0)

# Manual 95% CI
se = le.std() / np.sqrt(len(le))
ci_low  = le.mean() - 1.96 * se
ci_high = le.mean() + 1.96 * se
d = (le.mean() - mu_0) / le.std()

ttest_report(
    f"One-sample: Is mean LE = {mu_0}?",
    t, p, len(le),
    ci=(ci_low, ci_high),
    d=d
)
print(f"  Sample mean = {le.mean():.2f} yrs  (μ₀ = {mu_0})")

In [ ]:
# ── Two-sample Welch's t-test ──────────────────────────────────────────
# H₀: North Africa LE = Sub-Saharan Africa LE
north     = df[df["region"] == "North Africa"]["life_expectancy"]
sub_sah   = df[df["region"] != "North Africa"]["life_expectancy"]

t2, p2 = ttest_ind(north, sub_sah, equal_var=False)   # Welch's
d2 = (north.mean() - sub_sah.mean()) / np.sqrt(
    (north.std()**2 + sub_sah.std()**2) / 2
)

ttest_report(
    "Two-sample: North Africa vs Sub-Saharan LE (Welch's)",
    t2, p2,
    n=f"{len(north)} vs {len(sub_sah)}",
    d=d2
)
print(f"  North Africa mean  = {north.mean():.2f} yrs")
print(f"  Sub-Saharan mean   = {sub_sah.mean():.2f} yrs")

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
groups = {"North Africa": north, "Sub-Saharan Africa": sub_sah}
colors = {"North Africa": "#E74C3C", "Sub-Saharan Africa": "#3498DB"}
for i, (label, data) in enumerate(groups.items()):
    ax.boxplot(data, positions=[i], patch_artist=True,
               boxprops=dict(facecolor=colors[label], alpha=0.7),
               medianprops=dict(color="black", lw=2),
               widths=0.4)
    ax.scatter(np.full(len(data), i) + np.random.uniform(-0.08, 0.08, len(data)),
               data, alpha=0.5, color=colors[label], s=30, zorder=3)
ax.set_xticks([0, 1])
ax.set_xticklabels(["North Africa", "Sub-Saharan Africa"])
ax.set_ylabel("Life Expectancy (yrs)")
ax.set_title(f"Two-Sample t-Test: p = {p2:.4f}, d = {d2:.2f}", fontweight="bold")
ax.text(0.5, 0.95, f"t = {t2:.2f}, {'★ Significant' if p2<0.05 else 'Not significant'}",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=10, color="green" if p2<0.05 else "gray")
plt.tight_layout()
plt.savefig("week6_ttest.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Paired t-test ─────────────────────────────────────────────────────
# Simulate: vaccination rates measured in 2015 and 2022 for same countries
np.random.seed(7)
vax_2015 = df["vaccination_dtp3"] - np.random.uniform(5, 15, len(df))  # 2015 was lower
vax_2022 = df["vaccination_dtp3"]

t3, p3 = ttest_rel(vax_2022, vax_2015)
diff = vax_2022 - vax_2015
d3 = diff.mean() / diff.std()

ttest_report(
    "Paired: DTP3 Vaccination — 2015 vs 2022",
    t3, p3, len(diff), d=d3
)
print(f"  Mean improvement = {diff.mean():.2f} percentage points")

---
## 4. Chi-Square Tests

Used when both variables are **categorical**.

| Test | H₀ |
|---|---|
| **χ² goodness-of-fit** | Observed distribution = expected distribution |
| **χ² independence** | Two categorical variables are independent |

In [ ]:
# Chi-square test of independence
# Question: Is high vaccination rate independent of region?

# Create binary variables
df["high_vax"]   = (df["vaccination_dtp3"] >= 80).map({True: "High (≥80%)", False: "Low (<80%)"})
df["north_flag"] = df["region"].apply(lambda r: "North Africa" if r == "North Africa" else "Other")

# Contingency table
ct = pd.crosstab(df["north_flag"], df["high_vax"])
print("Contingency Table:")
print(ct)
print()

chi2_stat, p_chi, dof, expected = chi2_contingency(ct)

print(f"Chi-square Test of Independence")
print(f"  χ² = {chi2_stat:.4f}")
print(f"  p  = {p_chi:.4f}")
print(f"  df = {dof}")
print(f"  Expected frequencies:\n{np.round(expected, 1)}")
sig = "✓ Significant" if p_chi <= 0.05 else "✗ Not significant"
print(f"\n  Decision (α=0.05): {sig}")
print("  Interpretation: ", end="")
if p_chi <= 0.05:
    print("Vaccination rate is NOT independent of region.")
else:
    print("Vaccination rate appears independent of this regional grouping.")

# Visualise contingency table as grouped bar
fig, ax = plt.subplots(figsize=(7, 4))
ct.plot(kind="bar", ax=ax, color=["#E74C3C", "#3498DB"],
        edgecolor="white", alpha=0.85)
ax.set_title(f"Vaccination Level by Region Group\n"
             f"χ²={chi2_stat:.2f}, p={p_chi:.3f} ({sig})", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Vaccination Level")
plt.tight_layout()
plt.savefig("week6_chisquare.png", bbox_inches="tight")
plt.show()

---
## 5. One-Way ANOVA

**ANOVA** (Analysis of Variance) tests whether the means of **three or more groups** are equal.

- **H₀:** All group means are equal (μ₁ = μ₂ = μ₃ = … = μₖ)
- **H₁:** At least one group mean differs

ANOVA only tells you *that* a difference exists — **post-hoc tests** (Tukey HSD) identify *which* groups differ.

In [ ]:
from scipy.stats import f_oneway

# One-way ANOVA: does LE differ across regions?
region_groups = [grp["life_expectancy"].values
                 for _, grp in df.groupby("region")]

F, p_anova = f_oneway(*region_groups)

print("=" * 50)
print("ONE-WAY ANOVA: Life Expectancy across Regions")
print(f"  F-statistic = {F:.4f}")
print(f"  p-value     = {p_anova:.4f}")
sig = "✓ Significant" if p_anova <= 0.05 else "✗ Not significant"
print(f"  Decision    : {sig}")
print()

# Group means
print("Group means:")
for region, grp in df.groupby("region"):
    print(f"  {region:<20} n={len(grp):2d}  mean={grp['life_expectancy'].mean():.1f}")
print("=" * 50)

# Post-hoc: Tukey HSD (requires statsmodels)
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    tukey = pairwise_tukeyhsd(df["life_expectancy"], df["region"], alpha=0.05)
    print("\nTukey HSD Post-Hoc Results:")
    print(tukey)
except ImportError:
    print("(Install statsmodels for Tukey HSD: pip install statsmodels)")

In [ ]:
# ANOVA visualisation
fig, ax = plt.subplots(figsize=(11, 5))

PALETTE = {
    "West Africa": "#3498DB", "East Africa": "#2ECC71",
    "North Africa": "#E74C3C", "Central Africa": "#9B59B6",
    "Southern Africa": "#F39C12",
}

region_order = df.groupby("region")["life_expectancy"].mean().sort_values().index
positions = np.arange(len(region_order))

for i, region in enumerate(region_order):
    grp = df[df["region"] == region]["life_expectancy"]
    bp  = ax.boxplot(grp, positions=[i], widths=0.35,
                     patch_artist=True,
                     boxprops=dict(facecolor=PALETTE[region], alpha=0.7),
                     medianprops=dict(color="black", lw=2),
                     flierprops=dict(marker="o", markersize=5,
                                     markerfacecolor=PALETTE[region]))
    ax.scatter(np.full(len(grp), i) + np.random.uniform(-0.1, 0.1, len(grp)),
               grp, alpha=0.5, color=PALETTE[region], s=30, zorder=3)
    ax.text(i, grp.mean() + 1.2, f"{grp.mean():.1f}",
            ha="center", fontsize=8, fontweight="bold")

ax.axhline(df["life_expectancy"].mean(), color="gray", ls=":",
           lw=1.5, label=f"Grand mean: {df['life_expectancy'].mean():.1f}")
ax.set_xticks(positions)
ax.set_xticklabels(region_order, rotation=15, ha="right")
ax.set_ylabel("Life Expectancy (years)")
ax.set_title(f"One-Way ANOVA: F={F:.2f}, p={p_anova:.4f} → {sig}",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("week6_anova.png", bbox_inches="tight")
plt.show()

---
## 6. Confidence Intervals

A **95% confidence interval** means: if we repeated the sampling procedure 100 times,  
approximately 95 of the intervals would contain the true population parameter.

**Formula (large sample):**
$$CI = \bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$$

For z₉₅ = 1.96, z₉₉ = 2.576

In [ ]:
def confidence_interval(data, confidence=0.95):
    """Compute confidence interval using t-distribution."""
    n    = len(data)
    mean = np.mean(data)
    se   = stats.sem(data)
    h    = se * stats.t.ppf((1 + confidence) / 2, df=n-1)
    return mean, mean - h, mean + h

# CI for each region's life expectancy
print("95% Confidence Intervals for Mean Life Expectancy by Region")
print("-" * 65)
region_ci = []
for region in df["region"].unique():
    data = df[df["region"] == region]["life_expectancy"]
    mean, lo, hi = confidence_interval(data)
    region_ci.append({"region": region, "mean": mean, "ci_low": lo, "ci_high": hi, "n": len(data)})
    print(f"  {region:<20} n={len(data):2d}  {mean:.1f}  [{lo:.1f}, {hi:.1f}]")

region_ci_df = pd.DataFrame(region_ci).sort_values("mean")

# Visualise CIs
fig, ax = plt.subplots(figsize=(9, 4))
y_pos = np.arange(len(region_ci_df))

ax.barh(y_pos, region_ci_df["mean"],
        xerr=[
            region_ci_df["mean"] - region_ci_df["ci_low"],
            region_ci_df["ci_high"] - region_ci_df["mean"]
        ],
        color=[PALETTE[r] for r in region_ci_df["region"]],
        edgecolor="white", alpha=0.85, capsize=5, error_kw={"lw": 2})

ax.set_yticks(y_pos)
ax.set_yticklabels(region_ci_df["region"])
ax.set_xlabel("Mean Life Expectancy (years)")
ax.set_title("95% Confidence Intervals — Life Expectancy by Region",
             fontweight="bold")
ax.axvline(df["life_expectancy"].mean(), color="gray", ls="--", lw=1.2,
           label=f"Overall mean {df['life_expectancy'].mean():.1f}")
ax.legend()
plt.tight_layout()
plt.savefig("week6_ci.png", bbox_inches="tight")
plt.show()

---
## 7. Effect Size

Statistical significance tells you whether an effect exists.  
**Effect size** tells you how *large* the effect is — practically important with large samples.

### Cohen's d (two groups)
$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$$

| Cohen's d | Interpretation |
|---|---|
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

### Eta-squared (η²) for ANOVA
$$\eta^2 = \frac{SS_{between}}{SS_{total}}$$  

Proportion of variance explained by group membership.

In [ ]:
# Cohen's d for all region pairs vs North Africa
north_le = df[df["region"] == "North Africa"]["life_expectancy"]

print("Cohen's d: North Africa vs each other region")
print("-" * 50)
for region in df["region"].unique():
    if region == "North Africa":
        continue
    other = df[df["region"] == region]["life_expectancy"]
    pooled_std = np.sqrt((north_le.std()**2 + other.std()**2) / 2)
    d = (north_le.mean() - other.mean()) / pooled_std
    mag = "small" if abs(d)<0.5 else "medium" if abs(d)<0.8 else "large"
    print(f"  {region:<20}  d = {d:+.3f}  ({mag})")

# Eta-squared for ANOVA
grand_mean = df["life_expectancy"].mean()
SS_between = sum(
    len(grp) * (grp["life_expectancy"].mean() - grand_mean)**2
    for _, grp in df.groupby("region")
)
SS_total   = sum((df["life_expectancy"] - grand_mean)**2)
eta_sq     = SS_between / SS_total

print(f"\nEta-squared (η²) = {eta_sq:.4f}")
print(f"  → Region explains {eta_sq*100:.1f}% of variance in life expectancy")

---
## 8. Lab Exercises

### Lab 1: One-Sample t-Test
Test whether the mean `water_access` across Africa differs from the WHO benchmark of 70%.  
State H₀ and H₁, compute the test, and interpret the result.

In [ ]:
# Lab 1
# H₀: mean water_access = 70
# H₁: mean water_access ≠ 70
benchmark = 70.0
# TODO: ttest_1samp + interpretation

### Lab 2: Chi-Square Independence Test
Create a binary variable `high_water` (water_access ≥ 65).  
Create a binary variable `high_le` (life_expectancy ≥ 63).  
Test whether these two variables are independent.

In [ ]:
# Lab 2
df["high_water"] = (df["water_access"] >= 65).astype(int)
df["high_le"]    = (df["life_expectancy"] >= 63).astype(int)
# TODO: pd.crosstab → chi2_contingency → interpret

### Lab 3: ANOVA on a Different Variable
Run a one-way ANOVA to test whether `infant_mortality` differs across the 5 regions.  
Compute eta-squared and visualise with a violin plot showing means.

In [ ]:
# Lab 3
region_groups_im = [grp["infant_mortality"].values
                    for _, grp in df.groupby("region")]
# TODO: f_oneway + eta-squared + violin plot

---
## 9. Assignment — Week 6

**Problem 1 (25 pts):** Test whether the mean `health_expenditure` differs between:  
- (a) Countries with GDP per capita above vs below the median  
- (b) Countries with high vaccination (≥80%) vs low vaccination  

For each: state hypotheses, run Welch's t-test, compute Cohen's d, visualise, and interpret.

**Problem 2 (25 pts):** Create a `vax_category` variable: "Low" (<60%), "Medium" (60–80%), "High" (>80%).  
Run a one-way ANOVA on `infant_mortality` across these three groups.  
Report F, p, η², and a box plot with group means labelled.

**Problem 3 (25 pts):** Build a complete hypothesis testing pipeline function:  
```python
def test_group_difference(df, metric, group_col, group_a, group_b, alpha=0.05):
    """Run Welch's t-test and return a summary dict with t, p, d, CI, and decision."""
```
Call it for at least 3 different combinations.

**Problem 4 (25 pts):** Write a 200-word explanation of why **statistical significance ≠ practical significance**, using an example from the Africa Health dataset.

In [ ]:
# Problem 1
pass  # TODO

In [ ]:
# Problem 2
pass  # TODO

In [ ]:
# Problem 3
def test_group_difference(df, metric, group_col, group_a, group_b, alpha=0.05):
    # TODO
    pass

**Problem 4 — Written response (200 words):**

> *Type your answer here.*

---
## Summary — Week 6

| Concept | Key Point |
|---|---|
| Probability distributions | Know which distribution fits your data before testing |
| Hypothesis testing | H₀ / H₁ → test statistic → p-value → decision |
| One-sample t-test | Compare sample mean to known value |
| Two-sample t-test | Compare means of independent groups (use Welch's) |
| Paired t-test | Compare before/after on same subjects |
| Chi-square | Test independence of two categorical variables |
| One-way ANOVA | Compare means across ≥ 3 groups |
| Confidence interval | Range likely to contain the true parameter |
| Effect size | Cohen's d (two groups); η² (ANOVA) — practical significance |

**Next:** Week 7 — Regression Analysis